# Part III — Conditioning and Stability

## Trefethen & Bau, *Numerical Linear Algebra* (1997) — Lecture 12–19

这是《Numerical Linear Algebra》读书笔记的第 3 册。目标不是摘要原书，而是把每一讲整理成一份**可以独立读懂的数值线性代数讲义**，再把它映射到现代 ML systems。

每一讲尽量保持同一结构：数学对象 → 关键公式 → 几何/算法解释 → numerical stability → ML systems mapping → Python experiment。

本册自洽：下面的 setup cell 提供全部依赖，按顺序 run all 即可。

原书 PDF：https://www.stat.uchicago.edu/~lekheng/courses/309/books/Trefethen-Bau.pdf

---

**本系列共 6 册**（Trefethen & Bau, *Numerical Linear Algebra*, 40 Lectures）

| | |
|---|---|
| Part I | [Fundamentals](01_fundamentals.ipynb) |
| Part II | [QR Factorization and Least Squares](02_qr_least_squares.ipynb) |
| Part III | [Conditioning and Stability](03_conditioning_stability.ipynb) |
| Part IV | [Systems of Equations](04_systems_of_equations.ipynb) |
| Part V | [Eigenvalues](05_eigenvalues.ipynb) |
| Part VI | [Iterative Methods](06_iterative_methods.ipynb) |

索引与阅读顺序见 [00_index.ipynb](00_index.ipynb)。


In [1]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import scipy.linalg as sla
    import scipy.sparse.linalg as spla
    SCIPY=True
except Exception:
    SCIPY=False
rng=np.random.default_rng(7)
np.set_printoptions(precision=5,suppress=True)

def relerr(a,b):
    return np.linalg.norm(a-b)/max(np.linalg.norm(b),1e-30)

def stable_rank(A):
    s=np.linalg.svd(A,compute_uv=False)
    return np.sum(s*s)/(s[0]*s[0])

def spectral_norm_power(A,steps=30,seed=0):
    r=np.random.default_rng(seed)
    v=r.normal(size=A.shape[1])
    v/=np.linalg.norm(v)
    for _ in range(steps):
        v=A.T@(A@v)
        v/=np.linalg.norm(v)
    return np.linalg.norm(A@v)

def make_cond(n,kappa,seed=0):
    r=np.random.default_rng(seed)
    Q1,_=np.linalg.qr(r.normal(size=(n,n)))
    Q2,_=np.linalg.qr(r.normal(size=(n,n)))
    s=np.geomspace(1,1/kappa,n)
    return Q1@np.diag(s)@Q2.T

print(f'NumPy {np.__version__} | SciPy {SCIPY}')

NumPy 2.4.2 | SciPy True


## Notation / 贯穿全书的符号

我们主要讨论实矩阵；复数情形把转置 $A^T$ 换成共轭转置 $A^*$。

- $A\in\mathbb{R}^{m\times n}$
- 向量 2-norm：

$$
\Vert x\Vert_2=\sqrt{x^Tx}
$$

- induced matrix 2-norm：

$$
\Vert A\Vert_2=\max_{x\neq0}\frac{\Vert Ax\Vert_2}{\Vert x\Vert_2}=\sigma_{\max}(A)
$$

- Frobenius norm：

$$
\Vert A\Vert_F^2=\sum_{ij}a_{ij}^2=\sum_i\sigma_i^2
$$

- condition number：

$$
\kappa_2(A)=\Vert A\Vert_2\Vert A^{-1}\Vert_2
=\frac{\sigma_{\max}}{\sigma_{\min}}
$$

- unit roundoff：记为 $u$。典型 floating-point model：

$$
\mathrm{fl}(a\circ b)=(a\circ b)(1+\delta),\qquad |\delta|\lesssim u.
$$

计算机算一次加减乘除，不会得到精确的 $a\circ b$，而是得到一个带相对误差的结果。逐项：

- $\mathrm{fl}(\cdot)$：floating-point，机器实际算出来的数；
- $a\circ b$：一次精确运算（$\circ$ 是 $+,-,\times,/$）；
- $\delta$：这次运算引入的相对误差；
- $u$：unit roundoff，这种格式「一次正确舍入」的相对误差上限。

左边是机器结果，右边是「真值再乘 $1+\delta$」。约束的是**相对误差**，不是绝对误差：真值若是 $1.0$，FP32 下次运算大约落在 $1\pm 6\times 10^{-8}$，不会无缘无故错到 $1.01$。

常见 $u$：

- FP32：$u\approx 2^{-24}\approx 6\times 10^{-8}$
- FP16：$u\approx 2^{-11}\approx 5\times 10^{-4}$
- BF16：$u\approx 2^{-8}\approx 4\times 10^{-3}$

一次运算只错 $u$。病态问题可能把这个 $u$ 放大成 $\kappa u$ 量级的解误差。所以不要把「格式很粗」「矩阵很病态」「算法多放大了 rounding」三件事混成一句“数值不稳”。

### 区分

不要把下面三个问题混在一起：

1. **operator amplification**：$\Vert A\Vert$ 大不大？
2. **problem conditioning**：$A^{-1}$ 是否敏感？
3. **algorithm stability**：实现是否额外放大 rounding error？

现代 ML numerics 中，大量争论其实是把这三个层次混在了一起。

# Lecture 12 — Conditioning and Condition Numbers

### 1. Conditioning 是函数的局部 sensitivity

把问题写成 $y=f(x)$。局部绝对 condition number 可理解为 Jacobian norm：

$$
\kappa_{\mathrm{abs}}(x)=\Vert Df(x)\Vert.
$$

相对 condition number 则把输入输出 scale 加进去。

### 2. Linear solve 的一阶 perturbation

$$
Ax=b.
$$

扰动后

$$
(A+\Delta A)(x+\Delta x)=b+\Delta b.
$$

忽略二阶项 $\Delta A\Delta x$：

$$
A\Delta x\approx\Delta b-\Delta A x,
$$

于是

$$
\Delta x\approx A^{-1}(\Delta b-\Delta A x).
$$

取 norm：

$$
\frac{\Vert \Delta x\Vert}{\Vert x\Vert}
\lesssim
\kappa(A)
\left(
\frac{\Vert \Delta A\Vert}{\Vert A\Vert}
+
\frac{\Vert \Delta b\Vert}{\Vert b\Vert}
\right).
$$

这就是 condition number 的工程意义：input perturbation 的相对误差可能被 $\kappa(A)$ 放大。

### 3. SVD interpretation

$$
\Vert A^{-1}\Vert_2=1/\sigma_{\min}(A),
$$

所以最危险的是 near-null direction。

### 4. ML mapping

欠约束的 representation、近冗余 feature、flat Hessian directions，本质上都在制造小 singular/eigen directions。

In [12]:
A=make_cond(30,1e8,10)
xt=rng.normal(size=30)
b=A@xt
x0=np.linalg.solve(A,b)
db=rng.normal(size=30)
db*=1e-8*np.linalg.norm(b)/np.linalg.norm(db)
x1=np.linalg.solve(A,b+db)
rb=np.linalg.norm(db)/np.linalg.norm(b)
rx=relerr(x1,x0)
print(f'cond {np.linalg.cond(A)} relative b perturbation {rb} relative x change {rx} amplification {rx / rb}')

cond 100000000.04838386 relative b perturbation 1.0000000000000002e-08 relative x change 0.016511927077851218 amplification 1651192.7077851214


**ML numerics 自测**

**Q1.** condition number 本质在量什么？

**A.** 问题 $y=f(x)$ 的局部灵敏度。绝对条件数是 $\Vert Df(x)\Vert$；相对条件数再把输入输出 scale 算进去。

**Q2.** 线性方程组里 $\kappa(A)$ 怎么进 bound？

**A.** $\Vert\Delta x\Vert/\Vert x\Vert\lesssim\kappa(A)\,(\Vert\Delta A\Vert/\Vert A\Vert+\Vert\Delta b\Vert/\Vert b\Vert)$。相对输入误差最多被 $\kappa$ 放大。

**Q3.** SVD 下最危险的方向是哪条？

**A.** $\Vert A^{-1}\Vert_2=1/\sigma_{\min}$，near-null direction。欠约束 feature / flat Hessian 都在制造这种方向。

**Q4.** 监控 $\kappa$ 是在监控算法，还是监控问题？

**A.** 监控问题本身。算法再 stable，也只能保证 backward error 是 $O(u)$；forward error 仍可到 $\kappa u$。


# Lecture 13 — Floating Point Arithmetic

### 1. Floating-point model

一个 normalized floating-point number 可以抽象为

$$
x=\pm(1.b_1b_2\ldots b_p)_2\,2^e.
$$

有限 mantissa 意味着相邻 representable numbers 间距随 scale 变化。

通常用 unit roundoff $u$ 表示一次正确 rounding 的相对误差规模。对一个已经算出来的实数做舍入：

$$
\mathrm{fl}(x)=x(1+\delta),\qquad |\delta|\le u.
$$

对一次精确算术运算，标准模型写成

$$
\mathrm{fl}(a\circ b)=(a\circ b)(1+\delta),\qquad |\delta|\lesssim u.
$$

$\mathrm{fl}(\cdot)$ 是机器实际写出的数，$a\circ b$ 是精确加减乘除，$\delta$ 是这次运算的相对误差。约束的是相对误差，不是绝对误差。

常见 $u$：FP32 $\approx 6\times 10^{-8}$，FP16 $\approx 5\times 10^{-4}$，BF16 $\approx 4\times 10^{-3}$。一次运算只引入 $u$；若问题的 $\kappa$ 很大，解的 forward error 可以到 $\kappa u$ 量级。算法 stable 只保证「没有把 rounding 额外放大」，不保证病态问题仍有精确解。

### 2. Catastrophic cancellation

若 $x\approx y$，计算

$$
z=x-y
$$

时，输入本身的微小相对误差可能在小 residual 中占很大比例。

例如本来有

$$
x=1.0000001,\quad y=1.0000000,
$$

差值只有 $10^{-7}$。如果低精度根本无法分辨这两个数，结果直接变成 0。

### 3. BF16 vs FP16 的不同风险

- FP16：mantissa 相对多，但 exponent range 小，容易 overflow/underflow；
- BF16：exponent 类似 FP32，但 mantissa 很粗，容易丢掉小 relative differences。

### 4. ML high-risk operations

- long reduction；
- variance / RMSNorm statistics；
- softmax logits；
- reciprocal / sqrt；
- Gram matrices；
- small-pivot factorization。

In [13]:
for dt in [np.float16,np.float32,np.float64]:
    f=np.finfo(dt)
    print(f'{dt.__name__} eps {f.eps} tiny {f.tiny} max {f.max}')

float16 eps 0.000977 tiny 6.104e-05 max 6.55e+04
float32 eps 1.1920929e-07 tiny 1.1754944e-38 max 3.4028235e+38
float64 eps 2.220446049250313e-16 tiny 2.2250738585072014e-308 max 1.7976931348623157e+308


**ML numerics 自测**

**Q1.** $\mathrm{fl}(a\circ b)=(a\circ b)(1+\delta)$ 约束的是什么？

**A.** 一次运算的相对误差 $|\delta|\lesssim u$，不是绝对误差。$u$：FP32 $\approx 6\times 10^{-8}$，FP16 $\approx 5\times 10^{-4}$，BF16 $\approx 4\times 10^{-3}$。

**Q2.** 什么时候相对误差模型会突然崩？

**A.** catastrophic cancellation：$x\approx y$ 时 $x-y$ 把已有误差抬成主导项。低精度甚至根本分不出 $1.0000001$ 和 $1$。

**Q3.** BF16 和 FP16 的风险有何不同？

**A.** FP16 mantissa 较多但 exponent 窄，怕 overflow/underflow；BF16 exponent 像 FP32，但 mantissa 粗，怕丢掉小相对差。

**Q4.** ML 里哪些 op 最吃 $u$？

**A.** 长 reduction、RMSNorm/variance、softmax、reciprocal/sqrt、Gram、$A^TA$、小 pivot 分解。


# Lecture 14 — Stability

### 1. Forward error

算法输出 $\hat y$，真正答案 $y=f(x)$：

$$
\text{forward error}=\Vert \hat y-y\Vert.
$$

### 2. Backward error

找最小 $\Delta x$，使

$$
\hat y=f(x+\Delta x).
$$

如果所需 $\Delta x$ 与 machine precision 同量级，我们说算法 backward stable。

### 3. Residual 不是 forward error

对 linear solve，

$$
r=b-A\hat x.
$$

因为

$$
A(x-\hat x)=r,
$$

所以

$$
x-\hat x=A^{-1}r.
$$

因此

$$
\Vert x-\hat x\Vert \le \Vert A^{-1}\Vert \Vert r\Vert.
$$

如果 $A^{-1}$ 很大，小 residual 仍然可能对应大 solution error。

### 4. Deployment parity mapping

只看 tensor residual/output diff 不够。至少要知道：

- local error 多大；
- downstream operator gain 多大；
- task output 对该方向 sensitivity 多大。

In [14]:
A=make_cond(25,1e12,12)
xt=rng.normal(size=25)
b=A@xt
x=np.linalg.solve(A,b)
print(f'relative residual {relerr(A @ x, b)}')
print(f'forward error {relerr(x, xt)}')
print(f'cond {np.linalg.cond(A)}')

relative residual 1.39684738700378e-16
forward error 5.386772903402778e-06
cond 999992300445.1638


**ML numerics 自测**

**Q1.** forward error 和 backward error 各是什么？

**A.** forward：$\Vert\hat y-y\Vert$，答案错多少。backward：最小 $\Delta x$ 使 $\hat y=f(x+\Delta x)$。$\Delta x$ 与 $u$ 同量级就称 backward stable。

**Q2.** 为什么 residual 不是 forward error？

**A.** $r=b-A\hat x$ 时 $x-\hat x=A^{-1}r$，故 $\Vert x-\hat x\Vert\le\Vert A^{-1}\Vert\Vert r\Vert$。$A^{-1}$ 大时，小 residual 仍可对应大解误差。

**Q3.** 只看 output tensor diff 够不够做 deployment parity？

**A.** 不够。还要看 local error、downstream gain、以及 task 对该方向的 sensitivity。

**Q4.** stable 算法保证的是哪一种 error？

**A.** 保证的是小 backward error，不是小 forward error。


# Lecture 15 — More on Stability

### 1. “Backward stable + well-conditioned = accurate”

这是 numerical analysis 最重要的组合关系之一。

若算法 backward stable，意味着它实际上求解的是 nearby problem：

$$
\hat y=f(x+\Delta x),
\qquad
\frac{\Vert \Delta x\Vert}{\Vert x\Vert}=O(u).
$$

若问题的 relative condition number 为 $\kappa$，则一阶近似：

$$
\frac{\Vert \hat y-y\Vert}{\Vert y\Vert}
=O(\kappa u).
$$

所以误差可以拆成：

$$
\boxed{\text{problem sensitivity}}
\times
\boxed{\text{algorithm perturbation}}.
$$

### 2. 为什么这个分解适合 mixed precision

假设某个 kernel 引入局部 perturbation $\epsilon$，下游 Jacobian gain 为 $G$：

$$
\delta y\approx G\epsilon.
$$

这就是现代版的 condition × backward error 思维。

### 3. 一个有用的 sanity scale

若 FP32 $u\sim10^{-7}$，而 $\kappa\sim10^8$，那么 $\kappa u$ 已经接近 10。此时“算法很 stable”也不能保证 forward answer 有意义。

**ML numerics 自测**

**Q1.** “backward stable + well-conditioned = accurate” 怎么拆？

**A.** 算法给出 nearby problem：$\Vert\Delta x\Vert/\Vert x\Vert=O(u)$。问题相对条件数为 $\kappa$，则 $\Vert\hat y-y\Vert/\Vert y\Vert=O(\kappa u)$。

**Q2.** mixed precision 里这句话变成什么？

**A.** kernel 引入局部扰动 $\epsilon$，下游 Jacobian gain 为 $G$，则 $\delta y\approx G\epsilon$。仍是 condition $\times$ backward error。

**Q3.** FP32、$u\sim 10^{-7}$、$\kappa\sim 10^8$ 说明什么？

**A.** $\kappa u$ 已接近 10。“算法很 stable”也不能保证 forward answer 有意义。

**Q4.** 该分别报告哪两个盒子？

**A.** problem sensitivity 和 algorithm perturbation。混成一句“数值不稳”会选错修法。


# Lecture 16 — Stability of Householder Triangularization

### 1. Householder QR 的 backward stability 结论

设 floating-point 算法计算出 $\hat Q,\hat R$。核心结论可理解为存在一个很小的 $\Delta A$，使

$$
A+\Delta A=\hat Q\hat R,
\qquad
\frac{\Vert \Delta A\Vert}{\Vert A\Vert}=O(u)
$$

（更精确的 bound 还含维度常数）。

也就是说算法没有把 rounding error 变成一个远离原输入的问题。

### 2. 为什么 orthogonal transformations 有优势

每一步 reflector $H_k$ 的 norm 是 1：

$$
\Vert H_k\Vert_2=1.
$$

所以前面某一步产生的 perturbation 在后续正交变换中不会被指数放大。

### 3. 但 QR stable ≠ least squares 一定 accurate

如果 $A$ 接近 rank deficient，$R$ 的 diagonal 会出现极小值。QR 可能准确地揭示这个坏条件数，但 solve 本身仍然 sensitive。

**ML numerics 自测**

**Q1.** Householder QR 的 backward stability 结论是什么？

**A.** 存在 $\Delta A$ 使 $A+\Delta A=\hat Q\hat R$，且 $\Vert\Delta A\Vert/\Vert A\Vert=O(u)$。算法没有把 rounding 变成远离原问题的另一个问题。

**Q2.** 为什么正交变换帮得上忙？

**A.** 每步 $\Vert H_k\Vert_2=1$，前面的 perturbation 不会在后续反射里被指数放大。

**Q3.** QR stable 是否等于 least squares accurate？

**A.** 不等于。$A$ 接近亏秩时 $R$ 对角会出现极小值。QR 可能准确地报告坏条件，solve 仍然敏感。

**Q4.** 该监控 $\Vert\Delta A\Vert$ 还是 $\Vert\hat x-x\Vert$？

**A.** 先看 backward residual / $\Vert A-\hat Q\hat R\Vert$ 判断分解；再单独看 $\kappa(R)$ 判断后续 solve。


# Lecture 17 — Stability of Back Substitution

### 1. Back substitution

对 upper triangular $R$：

$$
Rx=b.
$$

最后一行先得到

$$
x_n=b_n/r_{nn},
$$

然后递推

$$
x_i=\frac{1}{r_{ii}}
\left(b_i-\sum_{j=i+1}^nr_{ij}x_j\right).
$$

### 2. Numerical risk

两个地方最敏感：

1. $r_{ii}$ 很小：division 放大误差；
2. 括号中的 subtraction cancellation。

对一个 backward-stable triangular solver，可以把计算结果解释为

$$
(R+\Delta R)\hat x=b,
\qquad
|\Delta R|\lesssim O(u)|R|.
$$

但如果 $R$ condition number 大，forward error 依旧可能很大。

### 3. ML mapping

Cholesky/QR/LU 最终都落到 triangular solve，因此“factorization 成功”不代表整个 solve 已经安全。

In [15]:
n=12
R=np.triu(rng.normal(size=(n,n)))
np.fill_diagonal(R,np.geomspace(1,1e-7,n))
xt=rng.normal(size=n)
b=R@xt
x32=np.linalg.solve(R.astype(np.float32),b.astype(np.float32)).astype(float)
print(f'cond(R) {np.linalg.cond(R)} FP32 forward error {relerr(x32, xt)} residual {relerr(R @ x32, b)}')

cond(R) 2.165313245317661e+24 FP32 forward error 1.5518121342003009e+22 residual 40001657452468.914


**ML numerics 自测**

**Q1.** 回代 $Rx=b$ 哪两步最危险？

**A.** $x_n=b_n/r_{nn}$：小 $r_{ii}$ 的除法放大误差；括号里 $b_i-\sum r_{ij}x_j$ 的 subtraction cancellation。

**Q2.** backward-stable triangular solve 实际在解什么？

**A.** $(R+\Delta R)\hat x=b$，且 $|\Delta R|\lesssim O(u)|R|$。$R$ 的 $\kappa$ 大时 forward error 仍可很大。

**Q3.** factorization 成功是否等于整个 solve 安全？

**A.** 不等于。Cholesky/QR/LU 最后都落到 triangular solve。分解成功只走完前半段。

**Q4.** ML 小系统 solve 该盯哪个量？

**A.** $|r_{ii}|$、$\kappa(R)$、residual，以及相对 FP64 的 forward error。


# Lecture 18 — Conditioning of Least Squares Problems

### 1. Least squares conditioning 比 square solve 更丰富

$$
x_*=(A^TA)^{-1}A^Tb=A^+b
$$

（full column rank）。如果 $\sigma_{\min}(A)$ 很小，$A^+$ norm 很大：

$$
\Vert A^+\Vert_2=1/\sigma_{\min}(A).
$$

### 2. Residual angle

令 fitted vector $p=Ax_*$，residual $r=b-p$。当 $b$ 几乎 orthogonal to column space 时，$p$ 很小而 residual 很大，relative parameter sensitivity 也会恶化。

因此 least-squares conditioning 不仅取决于 $\kappa(A)$，还取决于 $b$ 与 $\mathcal R(A)$ 的几何关系。

### 3. ML interpretation

对于 linear probe，如果 target 大部分不在 embedding span 中：

- residual 大；
- weak singular directions 决定的 coefficient 会非常不稳定；
- 不同 checkpoint 可以得到类似 prediction loss，却有截然不同的 parameter vector。

这就是为什么要同时看 prediction stability 和 representation/parameter identifiability。

**ML numerics 自测**

**Q1.** least squares 的 $\Vert A^+\Vert$ 由什么决定？

**A.** full column rank 时 $x_*=A^+b$，$\Vert A^+\Vert_2=1/\sigma_{\min}(A)$。小 $\sigma_{\min}$ 让参数对扰动极敏感。

**Q2.** conditioning 是否只取决于 $\kappa(A)$？

**A.** 不只。还取决于 $b$ 与 $\mathcal{R}(A)$ 的夹角：residual 大、$p=Ax_*$ 小时，相对参数灵敏度更差。

**Q3.** linear probe 为何能 loss 相近、参数却完全不同？

**A.** target 大部分不在 embedding span 里时，弱 singular 方向上的系数不稳定。预测可以像，parameter 可以不像。

**Q4.** 该同时报哪两个稳定性？

**A.** prediction stability 和 parameter / representation identifiability。


# Lecture 19 — Stability of Least Squares Algorithms

### 1. Normal equations 为什么危险

$$
A^TAx=A^Tb.
$$

SVD：

$$
A=U\Sigma V^T
\Rightarrow
A^TA=V\Sigma^2V^T.
$$

因此

$$
\kappa_2(A^TA)
=\frac{\sigma_1^2}{\sigma_n^2}
=\kappa_2(A)^2.
$$

这不是小常数差异，而是**把 condition number 平方**。

### 2. 三种常见路线

**Normal equations**

$$
A^TAx=A^Tb
$$

便宜、结构简单，但失去精度。

**QR**

$$
A=QR,\qquad Rx=Q^Tb
$$

通常是 dense least squares 的稳定默认。

**SVD**

$$
x=V\Sigma^+U^Tb
$$

最能处理 rank deficiency，也最贵。

### 3. 工程判断

如果你是在做 calibration / fitting，并且 feature covariance spectrum 很差，不要只因为 $A^TA$ 容易写就默认 normal equations。

In [16]:
m,n=120,12
z=rng.normal(size=(m,1))
A=np.hstack([z+10**(-j/2)*rng.normal(size=(m,1)) for j in range(n)])
xt=rng.normal(size=n)
b=A@xt+1e-8*rng.normal(size=m)
xq,*_=np.linalg.lstsq(A,b,rcond=None)
AtA=(A.astype(np.float32).T@A.astype(np.float32)).astype(np.float32)
Atb=(A.astype(np.float32).T@b.astype(np.float32)).astype(np.float32)
try:
    xn=np.linalg.solve(AtA,Atb).astype(float)
    ne=relerr(xn,xt)
except np.linalg.LinAlgError: ne=np.inf
print(f'cond(A) {np.linalg.cond(A)} cond(A^TA) {np.linalg.cond(A.T @ A)}')
print(f'lstsq parameter error {relerr(xq, xt)}')
print(f'FP32 normal equations error {ne}')

cond(A) 499214.1895554182 cond(A^TA) 249214805257.4885
lstsq parameter error 1.4065965363944927e-05
FP32 normal equations error 166.91529190333821


**ML numerics 自测**

**Q1.** normal equations 最关键的数值罪行是什么？

**A.** $\kappa_2(A^TA)=\kappa_2(A)^2$。不是小常数，是把条件数平方。

**Q2.** 三条路线怎么选？

**A.** normal equations：便宜但不稳。QR：$A=QR$、$Rx=Q^Tb$，dense LS 的默认稳定选择。SVD：$x=V\Sigma^+U^Tb$，最能处理亏秩，也最贵。

**Q3.** feature covariance spectrum 很差时，为什么不要图省事写 $A^TA$？

**A.** calibration / fitting 时 $A^TA$ 好写，但精度先死在平方后的 $\kappa$。

**Q4.** 该盯 $\kappa(A)$ 还是 $\kappa(A^TA)$？

**A.** 两者都看。若你走了 normal equations，真正决定参数误差的是 $\kappa(A^TA)$。
